### LOADING AND CONFIGURATION

In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
import joblib
import os

# --- CONFIGURATION ---
DATA_PATH = r'D:\PJT2\CMAPSSData' 
PROCESSED_PATH = r'../../processed_data' # Going back from models/global_FD004
RUL_CAP = 125 
WINDOW_SIZE = 60 

if not os.path.exists(PROCESSED_PATH):
    os.makedirs(PROCESSED_PATH)

cols = ['unit_number', 'time_in_cycles', 'setting_1', 'setting_2', 'setting_3'] + [f's{i}' for i in range(1, 22)]

print("Loading FD004 Raw Data...")
train_df = pd.read_csv(os.path.join(DATA_PATH, 'train_FD004.txt'), sep='\s+', header=None, names=cols)
test_df = pd.read_csv(os.path.join(DATA_PATH, 'test_FD004.txt'), sep='\s+', header=None, names=cols)
true_ruls = pd.read_csv(os.path.join(DATA_PATH, 'RUL_FD004.txt'), header=None, names=['RUL'])

Loading FD004 Raw Data...


### CALCULATING RUL

In [2]:
print("Calculating RUL...")
max_cycle = train_df.groupby('unit_number')['time_in_cycles'].max().reset_index()
max_cycle.columns = ['unit_number', 'max_cycle']
train_df = train_df.merge(max_cycle, on='unit_number', how='left')
train_df['RUL'] = train_df['max_cycle'] - train_df['time_in_cycles']
train_df.drop(columns=['max_cycle'], inplace=True)
train_df['RUL'] = train_df['RUL'].clip(upper=RUL_CAP)

Calculating RUL...


### FEATURES

In [3]:
dataset_stds = train_df.describe().loc['std']
constant_features = dataset_stds[dataset_stds == 0].index.tolist()
drop_labels = constant_features + ['unit_number', 'time_in_cycles', 'RUL']
features = [c for c in train_df.columns if c not in drop_labels]

print(f"Features selected ({len(features)}): {features}")

Features selected (24): ['setting_1', 'setting_2', 'setting_3', 's1', 's2', 's3', 's4', 's5', 's6', 's7', 's8', 's9', 's10', 's11', 's12', 's13', 's14', 's15', 's16', 's17', 's18', 's19', 's20', 's21']


### SCALING AND SMOOTHING

In [4]:
print("Scaling and Smoothing Data...")
scaler = MinMaxScaler()
train_df[features] = scaler.fit_transform(train_df[features])
test_df[features] = scaler.transform(test_df[features])

# Save Global Scaler
joblib.dump(scaler, os.path.join(PROCESSED_PATH, 'fd004_global_scaler.pkl'))

# Smooth (grouped by engine to prevent bleed)
train_df[features] = train_df.groupby('unit_number')[features].transform(lambda x: x.ewm(alpha=0.1).mean())
test_df[features] = test_df.groupby('unit_number')[features].transform(lambda x: x.ewm(alpha=0.1).mean())

Scaling and Smoothing Data...


### SEQUENCE GENERATION

In [5]:
def gen_sequence(id_df, seq_length, seq_cols):
    data_matrix = id_df[seq_cols].values
    num_elements = data_matrix.shape[0]
    for start, stop in zip(range(0, num_elements-seq_length), range(seq_length, num_elements)):
        yield data_matrix[start:stop, :]

def gen_labels(id_df, seq_length, label):
    data_matrix = id_df[label].values
    return data_matrix[seq_length:]

### TRAINING AND TEST SEQUENCE

In [6]:
X_train, y_train = [], []
for unit_id in train_df['unit_number'].unique():
    unit_data = train_df[train_df['unit_number'] == unit_id]
    for seq in gen_sequence(unit_data, WINDOW_SIZE, features):
        X_train.append(seq)
    for label in gen_labels(unit_data, WINDOW_SIZE, 'RUL'):
        y_train.append(label)

X_train = np.array(X_train)
y_train = np.array(y_train)

# Test Sequences (Only the last window per engine!)
X_test = []
for unit_id in test_df['unit_number'].unique():
    unit_data = test_df[test_df['unit_number'] == unit_id]
    if len(unit_data) >= WINDOW_SIZE:
        seq = unit_data[features].values[-WINDOW_SIZE:]
        X_test.append(seq)
    else:
        # Pad if engine lived less than 30 cycles
        seq = np.pad(unit_data[features].values, ((WINDOW_SIZE - len(unit_data), 0), (0, 0)), 'constant', constant_values=0)
        X_test.append(seq)

X_test = np.array(X_test)
y_test = true_ruls['RUL'].values

print(f"Final X_train Shape: {X_train.shape}")
print(f"Final X_test Shape: {X_test.shape}")

Final X_train Shape: (46309, 60, 24)
Final X_test Shape: (248, 60, 24)


### SAVING ARRAYS

In [7]:
np.save(os.path.join(PROCESSED_PATH, 'X_train_FD004_global.npy'), X_train)
np.save(os.path.join(PROCESSED_PATH, 'y_train_FD004_global.npy'), y_train)
np.save(os.path.join(PROCESSED_PATH, 'X_test_FD004_global.npy'), X_test)
np.save(os.path.join(PROCESSED_PATH, 'y_test_FD004_global.npy'), y_test)

print("Global FD004 Preprocessing Complete.")

Global FD004 Preprocessing Complete.
